In [1]:
import pandas as pd

In [2]:
# Load Eloundou exposure score and crosswalks
raw_df = pd.read_csv("../data/occ_level.csv")

onet_to_soc = pd.read_excel("../data/crosswalks/2019_to_SOC_Crosswalk.xlsx", 
                            sheet_name="O-NET-SOC 2019 Occupation Listi",
                            skiprows=3)
onet_to_nem = pd.read_excel("../data/crosswalks/nem-onet-to-soc-crosswalk.xlsx",
                            sheet_name="ONET to SOC crosswalk",
                            skiprows=4)
nem_to_cps = pd.read_excel("../data/crosswalks/nem-occcode-cps-crosswalk.xlsx",
                            sheet_name="NEM SOC CPS crosswalk",
                            skiprows=4)

In [3]:
onet_codes = raw_df[["O*NET-SOC Code"]]

# 923 O*NET 8-digit codes collapse to 798 6-digit SOC codes
onet_to_soc_merged = onet_codes.merge(
    onet_to_soc,
    left_on="O*NET-SOC Code",
    right_on="O*NET-SOC 2019 Code",
    how="left",
    indicator=True)

assert (onet_to_soc_merged["_merge"] == "left_only").sum() == 0, "Some O*NET codes failed to match SOC crosswalk"
onet_to_soc_merged = onet_to_soc_merged.drop(columns=["O*NET-SOC 2019 Code", "_merge"])

In [4]:
# 923 O*NET 8-digit codes collapse to 785 6-digit OEWS-SOC codes (level with employment data)
soc_to_oews_merged = onet_to_soc_merged.merge(
    onet_to_nem[["O*NET-SOC code", "NEM Code", "NEM title"]],
    left_on="O*NET-SOC Code",
    right_on="O*NET-SOC code",
    how="left",
    indicator=True)

assert (soc_to_oews_merged["_merge"] == "left_only").sum() == 0, "Some O*NET codes failed to match OEWS crosswalk"
soc_to_oews_merged = soc_to_oews_merged.drop(columns=["O*NET-SOC code", "_merge"])

In [5]:
# A few permanent aggregates are coarser in OEWS than CPS. So a standard crosswalk would merge
# 3 O*NET/SOC codes into a single OEWS code then fan it back out into a CPS code
# Thus they are manually assigned, see NEM to CPS crosswalk and 
# https://www.bls.gov/opub/mlr/2021/article/mapping-employment-projections-and-onet-data.htm

# Direct mappings bypass the OEWS level collapse, we can mantain 1:1 here
soc_to_cps_direct = {
    "13-1021": ("510", "Buyers and purchasing agents, farm products"),
    "13-1022": ("520", "Wholesale and retail buyers, except farm products"),
    "13-1023": ("530", "Purchasing agents, except wholesale, retail, and farm products"),
    "21-1011": ("2001", "Substance abuse and behavioral disorder counselors"),
    "21-1014": ("2004", "Mental health counselors"),
    "31-1121": ("3601", "Home health aides"),
    "31-1122": ("3602", "Personal care aides"),
}

is_aggregate = soc_to_oews_merged["2018 SOC Code"].isin(soc_to_cps_direct)

aggregate_rows = soc_to_oews_merged[is_aggregate].copy()
aggregate_rows["CPS code"] = aggregate_rows["2018 SOC Code"].map(lambda x: soc_to_cps_direct[x][0])
aggregate_rows["CPS occupation title"] = aggregate_rows["2018 SOC Code"].map(lambda x: soc_to_cps_direct[x][1])
aggregate_rows["manual_soc_to_cps"] = True

# Everything else
normal_rows = soc_to_oews_merged[~is_aggregate].merge(
    nem_to_cps[["National Employment Matrix code", "CPS code", "CPS occupation title"]],
    left_on="NEM Code",
    right_on="National Employment Matrix code",
    how="left",
    indicator=True)
normal_rows["manual_soc_to_cps"] = False

crosswalk = pd.concat([normal_rows, aggregate_rows], ignore_index=True)

assert (crosswalk["_merge"] == "left_only").sum() == 0, "Some OEWS codes failed to match CPS crosswalk"
assert crosswalk.shape[0] == 923, "Rowcount incorrect"
crosswalk = crosswalk.drop(columns=["National Employment Matrix code", "_merge"])

In [6]:
# Rename columns for consistency
crosswalk = crosswalk.rename(columns={
    "O*NET-SOC Code": "ONET Code",
    "O*NET-SOC 2019 Title": "ONET Title",
    "2018 SOC Code": "SOC Code",
    "2018 SOC Title": "SOC Title",
    "NEM Code": "OEWS Code",
    "NEM title": "OEWS Title",
    "CPS code": "CPS Code",
    "CPS occupation title": "CPS Title"})

crosswalk.to_csv("../data/crosswalks/final_crosswalk.csv", index=False)